# M8 · Feature EDA — bridge to Phase 4

**Goal:** pull `marts.v_ml_features_driver_behavior` into pandas, sanity-check distributions, correlations, tenant-level stratification. **No modeling yet.**

**Exit criterion:** you can identify (a) redundant features, (b) features with near-zero variance, (c) apparent natural segments in a 2-D PCA/UMAP projection.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — pull the feature table

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    df = pd.read_sql(text('SELECT * FROM marts.v_ml_features_driver_behavior'), conn)
print(df.shape); df.head()


## 3. Execute — EDA

In [ ]:
# Distributions
import matplotlib.pyplot as plt
num = df.select_dtypes('number')
num.describe().T


In [ ]:
# Correlation heatmap
import numpy as np
import matplotlib.pyplot as plt
corr = df.select_dtypes('number').corr()
fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im); plt.tight_layout(); plt.show()


In [ ]:
# Per-tenant distribution of a key feature
import matplotlib.pyplot as plt
if 'overspeed_per_100km' in df.columns and 'tenant_id' in df.columns:
    df.boxplot(column='overspeed_per_100km', by='tenant_id', figsize=(8,3))
    plt.suptitle(''); plt.tight_layout(); plt.show()


## 4. Inspect — write your notes here

### EDA notes

- …
- …
- …
